In [1]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()
BUCKET = "gs://retailsmart-lake-retailsmart-bd"

spark.conf.set("spark.sql.catalog.retailsmart", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.retailsmart.type", "hadoop")
spark.conf.set("spark.sql.catalog.retailsmart.warehouse", f"{BUCKET}/lakehouse")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailsmart.universal")

DataFrame[]

In [2]:
spark.conf.set("spark.sql.shuffle.partitions", "8")

raw = (spark.read.option("header", True).option("inferSchema", True)
       .csv(f"{BUCKET}/landing/olist/olist_orders_dataset.csv"))

estados = ["created", "approved", "invoiced", "processing",
           "shipped", "delivered", "unavailable", "canceled"]

checked = raw.withColumn(
    "motivo",
    F.when(F.col("order_id").isNull(), "order_id nulo")
     .when(F.col("order_purchase_timestamp").isNull(), "fecha de compra nula")
     .when(F.col("order_status").isNull() | ~F.col("order_status").isin(estados),
           "estado fuera de catálogo")
     .when(F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp"),
           "entrega anterior a la compra"))

malos = checked.filter(F.col("motivo").isNotNull())
buenos = (checked.filter(F.col("motivo").isNull()).drop("motivo")
          .dropDuplicates(["order_id"]))

n_leidas = raw.count()
n_malos = malos.count()
n_validos = buenos.count()
print("filas leídas:", n_leidas)
print("rechazadas por reglas:", n_malos)
print("duplicados eliminados:", (n_leidas - n_malos) - n_validos)
print("válidas:", n_validos)
malos.groupBy("motivo").count().show(truncate=False)

filas leídas: 99441
rechazadas por reglas: 0
duplicados eliminados: 0
válidas: 99441


+------+-----+
|motivo|count|
+------+-----+
+------+-----+



In [3]:
def con_metadatos(df):
    return (df.withColumn("_fuente", F.lit("olist"))
              .withColumn("_cargado_en", F.current_timestamp()))

(con_metadatos(buenos).writeTo("retailsmart.universal.silver_logistica_pedidos_v1")
    .using("iceberg").createOrReplace())
(con_metadatos(malos).writeTo("retailsmart.universal.cuarentena_logistica_pedidos_v1")
    .using("iceberg").createOrReplace())

spark.sql("SELECT COUNT(*) AS validos FROM retailsmart.universal.silver_logistica_pedidos_v1").show()
spark.sql("SELECT COUNT(*) AS rechazados FROM retailsmart.universal.cuarentena_logistica_pedidos_v1").show()

+-------+
|validos|
+-------+
|  99441|
+-------+



+----------+
|rechazados|
+----------+
|         0|
+----------+

